# 07 Multimodal Models

This notebook defines the deep multimodal modeling setup for early sepsis prediction. It aligns structured trajectories with temporal note windows, trains the configured fusion strategies locally, and writes experiment-ready artifacts for downstream evaluation.

## Architectural plan

- Structured encoder options: GRU or Transformer.
- Text encoder path: transformer embeddings when the configured model is available, with a deterministic hashing fallback for fully local runs.
- Fusion strategies: early fusion, late fusion, gated fusion, cross-modal attention.
- This notebook prepares aligned multimodal examples and trains the configured fusion variants locally.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy torch transformers

In [3]:
import pandas as pd

from src.training import prepare_multimodal_dataset, train_multimodal_models
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest

In [4]:
config['multimodal']

{'batch_size': 16,
 'hidden_dim': 128,
 'dropout': 0.2,
 'learning_rate': 0.0005,
 'epochs': 10,
 'device': 'auto',
 'weight_decay': 0.0001,
 'gradient_clip_norm': 1.0,
 'max_structured_steps': 48,
 'max_text_windows': 8,
 'max_tokens_per_window': 128,
 'text_embedding_dim': 768,
 'structured_encoder': 'gru',
 'fusion_strategies': ['early_fusion',
  'late_fusion',
  'gated_fusion',
  'cross_modal_attention']}

## Load structured and text artifacts

In [5]:
horizon = int(config['prediction']['horizons_hours'][0])
dataset_name = f'horizon_{horizon}h'
structured_path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
text_path = paths['processed_data_dir'] / '05_text_processing' / f'{dataset_name}_note_windows.csv'
assert structured_path.exists(), f'Missing structured dataset: {dataset_name}. Run 04_feature_engineering first.'
assert text_path.exists(), f'Missing note-window dataset: {dataset_name}_note_windows. Run 05_text_processing first.'

structured_df = pd.read_csv(structured_path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'])
text_df = pd.read_csv(text_path, parse_dates=['prediction_time', 'first_note_time', 'last_note_time'])
structured_df.shape, text_df.shape

((1268294, 90), (114663, 12))

## Prepare fixed-length multimodal examples

This stage aligns structured trajectories and note windows per ICU stay, normalizes structured inputs from the training split, and builds text embeddings using the configured transformer when available. If the transformer weights are not available locally, the pipeline falls back to a deterministic hashing encoder so training can still run on the machine.

In [6]:
prepared = prepare_multimodal_dataset(
    structured_df=structured_df,
    text_df=text_df,
    config=config,
)

prepared.stay_index[['split', 'sepsis3_label']].value_counts().rename('stay_count').reset_index()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,split,sepsis3_label,stay_count
0,train,0.0,25647
1,test,0.0,5528
2,val,0.0,5458
3,train,1.0,1228
4,test,1.0,299
5,val,1.0,287


In [7]:
pd.DataFrame([
    {'metric': 'n_stays', 'value': int(len(prepared.stay_index))},
    {'metric': 'structured_tensor_shape', 'value': str(prepared.structured_sequences.shape)},
    {'metric': 'text_tensor_shape', 'value': str(prepared.text_embeddings.shape)},
    {'metric': 'text_embedding_backend', 'value': prepared.text_embedding_backend},
    {'metric': 'n_structured_features', 'value': int(len(prepared.feature_columns))},
])

,metric,value
0,n_stays,38447
1,structured_tensor_shape,"(38447, 48, 75)"
2,text_tensor_shape,"(38447, 8, 768)"
3,text_embedding_backend,transformer:emilyalsentzer/Bio_ClinicalBERT
4,n_structured_features,75


## Train Fusion Variants

In [8]:
output_dir = paths['processed_data_dir'] / '07_multimodal_models'
training_output = train_multimodal_models(
    prepared=prepared,
    config=config,
    output_dir=output_dir,
    dataset_name=dataset_name,
)

results_key = f'{dataset_name}_multimodal_results'
history_key = f'{dataset_name}_multimodal_training_history'
plan_key = f'{dataset_name}_multimodal_experiment_plan'
results_df = training_output['artifacts'][results_key]
history_df = training_output['artifacts'][history_key]
experiment_plan_df = training_output['artifacts'][plan_key]
results_df

,dataset_name,split,model_name,structured_encoder,text_embedding_backend,device,n_features,n_examples,loss,auroc,...,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn
0,horizon_6h,val,early_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5745,0.191726,0.817728,...,0.370833,0.310105,0.337761,0.048971,0.972334,0.310105,89,151,5307,198
1,horizon_6h,test,early_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5827,0.204680,0.799145,...,0.355556,0.267559,0.305344,0.051462,0.973770,0.267559,80,145,5383,219
2,horizon_6h,val,late_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5745,0.192012,0.833686,...,0.354839,0.344948,0.349823,0.051915,0.967021,0.344948,99,180,5278,188
3,horizon_6h,test,late_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5827,0.204138,0.804024,...,0.298450,0.257525,0.276481,0.055749,0.967258,0.257525,77,181,5347,222
4,horizon_6h,val,gated_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5745,0.180157,0.841001,...,0.369919,0.317073,0.341463,0.048672,0.971601,0.317073,91,155,5303,196
5,horizon_6h,test,gated_fusion,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5827,0.196106,0.808292,...,0.352941,0.260870,0.300000,0.052045,0.974132,0.260870,78,143,5385,221
6,horizon_6h,val,cross_modal_attention,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5745,0.250975,0.587485,...,0.000000,0.000000,0.000000,0.060209,1.000000,0.000000,0,0,5458,287
7,horizon_6h,test,cross_modal_attention,gru,transformer:emilyalsentzer/Bio_ClinicalBERT,cuda,75,5827,0.252755,0.619064,...,0.000000,0.000000,0.000000,0.060781,1.000000,0.000000,0,0,5528,299


## Save Training Artifacts

In [9]:
saved_paths = save_dataframe_bundle(training_output['artifacts'], output_dir)
saved_paths

{'horizon_6h_early_fusion_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/07_multimodal_models/horizon_6h_early_fusion_val_predictions.csv',
 'horizon_6h_early_fusion_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/07_multimodal_models/horizon_6h_early_fusion_test_predictions.csv',
 'horizon_6h_late_fusion_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/07_multimodal_models/horizon_6h_late_fusion_val_predictions.csv',
 'horizon_6h_late_fusion_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/07_multimodal_models/horizon_6h_late_fusion_test_predictions.csv',
 'horizon_6h_gated_fusion_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/07_multimodal_models/horizon_6h_gated_fusion_val_predictions.csv',
 'horizon_6h_gated_fusion_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Pr

In [10]:
manifest_path = paths['manifests_dir'] / f'07_multimodal_models_{dataset_name}_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='07_multimodal_models',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'checkpoint_paths': training_output['checkpoint_paths'],
        'dataset_name': dataset_name,
        'fusion_strategies': config['multimodal']['fusion_strategies'],
        'device': training_output['device'],
        'text_embedding_backend': training_output['text_embedding_backend'],
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/07_multimodal_models_horizon_6h_manifest.json')

## Review checklist

Before clinical evaluation, review:

- Are the structured and text examples aligned on the same ICU stays?
- Which fusion strategies should get full training first?
- Does the selected text encoder fit the note window lengths?
- Should the structured encoder remain GRU or switch to Transformer for the main experiments?
- Are additional batching or masking utilities needed before large-scale training?

## Next notebook

`08_evaluation.ipynb` will consolidate model outputs into discrimination, calibration, and clinical utility metrics and create the paper-ready evaluation artifacts.